# Notebook 03 — Telecom translator and materialisation

This notebook translates the native synthetic telecom dataset into two physically
separable outputs:

- `SPEC-CORE/`: runtime-safe canonical telemetry and context.
- `SPEC-EVAL/`: fault, cause, ticket and gap truth.

The output path includes the contract version and a run ID, so a later contract
revision cannot overwrite an earlier translation.

In [ ]:
from pathlib import Path
import json
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive/anomaly_detection")
else:
    DEFAULT_DRIVE_ROOT = Path.cwd() / "anomaly_detection"

DRIVE_ROOT = Path(
    os.environ.get("ANOMALY_DETECTION_DRIVE_ROOT", str(DEFAULT_DRIVE_ROOT))
).expanduser()
CONTRACT_TAG = "v0.3"
CONTRACT_ROOT = DRIVE_ROOT / "contracts" / CONTRACT_TAG
PYTHON_SOURCE_ROOT = CONTRACT_ROOT / "python_src"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "milestone_1" / CONTRACT_TAG

print("Drive root:   ", DRIVE_ROOT)
print("Contract root:", CONTRACT_ROOT)
print("Output root:  ", OUTPUT_ROOT)

if not PYTHON_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Contract source not found at {PYTHON_SOURCE_ROOT}. Run Notebook 02 first."
    )
if str(PYTHON_SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE_ROOT))

## 1. Configuration

Leave the sampling controls empty for the full panel. For a development run, set a
time interval and/or a list of ONT IDs. Row-count truncation is deliberately not
supported because it produces partial entity histories.

In [ ]:
import resource
import tracemalloc
from datetime import datetime, timezone

from telemetry_adapters import NativeSelection, SyntheticGponAdapter
from telemetry_contract import canonical_frame_hash, sha256_file

TELECOM_SOURCE = Path(
    os.environ.get("ANOMALY_DETECTION_TELECOM_SOURCE", str(DRIVE_ROOT))
).expanduser()
RUN_ID = os.environ.get("ANOMALY_DETECTION_TELECOM_RUN_ID", "telecom_full_v1")

SAMPLE_START = os.environ.get("ANOMALY_DETECTION_SAMPLE_START") or None
SAMPLE_END = os.environ.get("ANOMALY_DETECTION_SAMPLE_END") or None
ENTITY_IDS = tuple(
    value.strip()
    for value in os.environ.get("ANOMALY_DETECTION_ENTITY_IDS", "").split(",")
    if value.strip()
)
BATCH_NATIVE_ROWS = int(os.environ.get("ANOMALY_DETECTION_BATCH_ROWS", "250000"))
MEMORY_BUDGET_GIB = float(os.environ.get("ANOMALY_DETECTION_MEMORY_BUDGET_GIB", "8"))

RUN_ROOT = OUTPUT_ROOT / "telecom" / RUN_ID
CORE_OUTPUT = RUN_ROOT / "SPEC-CORE"
EVAL_OUTPUT = RUN_ROOT / "SPEC-EVAL"

if RUN_ROOT.exists():
    raise FileExistsError(
        f"Refusing to overwrite immutable run {RUN_ROOT}. Choose a new RUN_ID."
    )
if not TELECOM_SOURCE.is_dir():
    raise FileNotFoundError(f"Telecom source not found: {TELECOM_SOURCE}")

selection = NativeSelection(
    sample_start=SAMPLE_START,
    sample_end=SAMPLE_END,
    entity_ids=ENTITY_IDS,
    batch_native_rows=BATCH_NATIVE_ROWS,
)
adapter = SyntheticGponAdapter(selection=selection)
inventory = adapter.discover(TELECOM_SOURCE)
print(inventory)
if not inventory.core_ready or not inventory.evaluation_ready:
    raise ValueError(inventory.notes)

## 2. Translate with bounded-memory instrumentation

`tracemalloc` measures a complete configured batch through widening and canonical
hashing. Tracing every allocation across all 69 million output observations would make
the full run needlessly slow, so full-run resident memory is measured by `ru_maxrss`.
That is the process high-water mark and never falls during a notebook session. Both
are recorded, with their scopes, because neither number alone tells the whole story.

In [ ]:
import gc
import pyarrow.parquet as pq

def ru_maxrss_gib():
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # macOS reports bytes; Linux (including Colab) reports KiB.
    return raw / (1024 ** 3) if sys.platform == "darwin" else raw * 1024 / (1024 ** 3)

# Trace one complete configured batch through the memory-heavy widening path.
rss_before_probe = ru_maxrss_gib()
panel_path = adapter._native_path(
    TELECOM_SOURCE, "reference_dataset.parquet"
)
panel_file = pq.ParquetFile(panel_path)
native_columns = set(panel_file.schema_arrow.names)
metric_fields = [
    field for field in adapter._metric_mapping() if field in native_columns
]
probe_columns = ["timestamp_utc", "ont_id", *metric_fields]
tracemalloc.start()
probe_native_rows = 0
probe_canonical_rows = 0
for batch in panel_file.iter_batches(
    batch_size=BATCH_NATIVE_ROWS, columns=probe_columns
):
    selected_probe = adapter._filter_native(batch.to_pandas())
    if selected_probe.empty:
        continue
    widened_probe = adapter.adapt_telemetry(
        selected_probe, cadence_seconds=900.0
    )
    canonical_frame_hash(
        widened_probe, sort_by=["event_ts", "entity_id", "metric_id"]
    )
    probe_native_rows = len(selected_probe)
    probe_canonical_rows = len(widened_probe)
    break
if probe_native_rows == 0:
    raise ValueError("The configured selection contains no probe rows.")
_, traced_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
del selected_probe, widened_probe, panel_file
gc.collect()

rss_before_full = ru_maxrss_gib()
report = adapter.materialise(TELECOM_SOURCE, CORE_OUTPUT, EVAL_OUTPUT)
rss_after = ru_maxrss_gib()

MEMORY_REPORT = {
    "measured_at_utc": datetime.now(timezone.utc).isoformat(),
    "tracemalloc_peak_gib": traced_peak_bytes / (1024 ** 3),
    "tracemalloc_scope": (
        "one configured native batch through long-format widening and "
        "canonical content hashing"
    ),
    "tracemalloc_probe_native_rows": probe_native_rows,
    "tracemalloc_probe_canonical_rows": probe_canonical_rows,
    "ru_maxrss_before_probe_gib": rss_before_probe,
    "ru_maxrss_before_full_gib": rss_before_full,
    "ru_maxrss_after_gib": rss_after,
    "ru_maxrss_scope": "entire full or selected materialisation",
    "ru_maxrss_semantics": "process high-water mark; it never falls in this kernel",
    "budget_gib": MEMORY_BUDGET_GIB,
    "budget_pass": rss_after <= MEMORY_BUDGET_GIB,
}
(RUN_ROOT / "memory_report.json").write_text(
    json.dumps(MEMORY_REPORT, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(MEMORY_REPORT, indent=2))
if not MEMORY_REPORT["budget_pass"]:
    raise MemoryError(
        "Peak resident memory exceeded the configured budget. "
        "Use a fresh kernel to distinguish current work from an earlier high-water mark."
    )
print(json.dumps(dict(report.row_counts), indent=2, sort_keys=True))

## 3. Source lineage manifest

The manifest pins every input file actually discovered. `tickets.csv` is explicitly
classified as evaluation-only.

In [ ]:
source_files = []
for relative_path in inventory.tables:
    path = TELECOM_SOURCE / relative_path
    source_files.append(
        {
            "path": relative_path,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "boundary": (
                "SPEC-EVAL only"
                if path.name == "tickets.csv"
                or path.name.startswith("gt_")
                or path.name == "fault_entity_intervals.csv"
                else "SPEC-CORE input"
            ),
        }
    )
SOURCE_MANIFEST = {
    "source_format": inventory.source_format,
    "source_version": inventory.source_version,
    "selection": {
        "sample_start": SAMPLE_START,
        "sample_end": SAMPLE_END,
        "entity_ids": list(ENTITY_IDS),
        "batch_native_rows": BATCH_NATIVE_ROWS,
    },
    "files": source_files,
}
(RUN_ROOT / "source_manifest.json").write_text(
    json.dumps(SOURCE_MANIFEST, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Pinned", len(source_files), "source files.")

## 4. Acceptance checks on values and table boundaries

This distinguishes:

1. a present row with a null value (`quality_code = invalid`);
2. an expected observation that is absent (`collection_gaps`);
3. an entity outside its valid service window (not a collection gap).

FEC values at the generator ceiling are marked `clipped`, not `measured`.

In [ ]:
import pandas as pd

core_tables = {
    path.stem for path in CORE_OUTPUT.glob("*.parquet")
} | ({"telemetry"} if (CORE_OUTPUT / "telemetry").is_dir() else set())
eval_tables = {path.stem for path in EVAL_OUTPUT.glob("*.parquet")}

assert "entity_service_windows" not in core_tables
assert "gt_ticket_links" in eval_tables
assert not any(name.startswith("gt_") for name in core_tables)

quality_counts = {}
exposure_values = {
    "telecom.link.fec_count": set(),
    "telecom.link.crc_errors": set(),
}
for part in sorted((CORE_OUTPUT / "telemetry").glob("part-*.parquet")):
    frame = pd.read_parquet(
        part, columns=["metric_id", "value", "quality_code", "exposure"]
    )
    for quality_code, count in frame["quality_code"].value_counts().items():
        quality_counts[str(quality_code)] = quality_counts.get(str(quality_code), 0) + int(count)
    for metric_id in exposure_values:
        values = frame.loc[frame["metric_id"].eq(metric_id), "exposure"].dropna().unique()
        exposure_values[metric_id].update(map(float, values))
    fec = frame.loc[frame["metric_id"].eq("telecom.link.fec_count")]
    assert fec.loc[fec["value"].eq(5_000_000), "quality_code"].eq("clipped").all()
    assert frame.loc[frame["value"].isna(), "quality_code"].eq("invalid").all()

core_manifest = json.loads((CORE_OUTPUT / "manifest.json").read_text(encoding="utf-8"))
cadence = float(core_manifest["cadence_seconds"])
assert exposure_values["telecom.link.fec_count"] == {2_488_000_000.0 * cadence}
assert exposure_values["telecom.link.crc_errors"] == {78_000_000.0 * cadence}

CHECK_REPORT = {
    "core_tables": sorted(core_tables),
    "eval_tables": sorted(eval_tables),
    "quality_counts": quality_counts,
    "exposure_unique_values": {
        key: sorted(values) for key, values in exposure_values.items()
    },
    "fec_constant_exposure_validated_by": "generator formula and dimensions",
    "tickets_boundary": "SPEC-EVAL only",
    "service_validity_storage": "entity_registry.valid_from/valid_to only",
}
(RUN_ROOT / "translation_checks.json").write_text(
    json.dumps(CHECK_REPORT, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(CHECK_REPORT, indent=2))
print("Notebook 03 complete:", RUN_ROOT)